# Project Report: Global Military Spending, GDP, and Arms Transfer Analysis

## 📊 Project Overview

This project provides a comprehensive analysis of global defense economics by integrating datasets from **SIPRI**, the **World Bank**, and international arms transfer records. The objective is to explore the relationship between economic power (GDP) and military capability, identify patterns in defense spending, and classify high-spending nations using machine learning.

### 🗝️ Key Features
- **Data Integration:** Merging multiple global datasets with automated standardization of country names.
- **Interactive Visuals:** Dynamic charts with sliders and buttons for deep exploration of trends and geographic distributions.
- **Advanced Analytics:** Regression modeling to study GDP-Spending correlation and Interactive Linear Regression to predict spending behavior.
- **Diverse Visualization Gallery:** Including Heatmaps, Choropleth Maps, Treemaps, Pairplots, and Doughnut charts.

### 🛠️ Tech Stack
- **Processing:** `Pandas`, `NumPy`, `Scikit-Learn`
- **Visualization:** `Matplotlib`, `Seaborn`, `Plotly Express`, `Ipywidgets`


In [1]:
# Import the libraries requested for the project.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets

# sklearn is used for scaling, regression, and classification.
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from IPython.display import display

pio.renderers.default = 'notebook_connected'
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (14, 7)
pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. Data Collection And Dataset Loading

This section loads the SIPRI military spending data, the World Bank GDP dataset, and the arms transfer dataset from the project folder.


In [2]:
# Locate the files. The notebook first looks in the current folder and then in a sibling folder if needed.
base_dir = Path.cwd()
fallback_dir = base_dir.parent / 'Project 1'

def locate_file(filename):
    candidates = [base_dir / filename, fallback_dir / filename]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Could not find {filename} in {base_dir} or {fallback_dir}')

sipri_path = locate_file('SIPRI-Milex-data-1949-2024_2.xlsx')
gdp_path = locate_file('WB_WDI_NY_GDP_MKTP_CD_WIDEF.csv')
arms_path = locate_file('import-export-top_1950-2026.csv')

print('SIPRI file:', sipri_path)
print('GDP file:', gdp_path)
print('Arms file:', arms_path)

# Load the raw source files.
sipri_raw = pd.read_excel(sipri_path, sheet_name='Current US$', header=5)
gdp_raw = pd.read_csv(gdp_path)

# The arms transfer file has a long metadata block, then one blank row, then the real header row.
arms_raw = pd.read_csv(arms_path, skiprows=10)

print('\nRaw dataset shapes:')
print('SIPRI:', sipri_raw.shape)
print('GDP:', gdp_raw.shape)
print('Arms:', arms_raw.shape)

SIPRI file: /Users/aryaghosh/Documents/Python with libraries (class)/Project/SIPRI-Milex-data-1949-2024_2.xlsx
GDP file: /Users/aryaghosh/Documents/Python with libraries (class)/Project/WB_WDI_NY_GDP_MKTP_CD_WIDEF.csv
Arms file: /Users/aryaghosh/Documents/Python with libraries (class)/Project/import-export-top_1950-2026.csv

Raw dataset shapes:
SIPRI: (193, 78)
GDP: (262, 106)
Arms: (140, 166)


## 2. Data Cleaning And Data Preparation

The next cells clean each source, reshape wide tables into long format, standardize country names, and merge the datasets for analysis.


In [3]:
# -----------------------------
# 1. Clean SIPRI military spending data
# -----------------------------

# Rename the first column to Country and drop unnamed helper columns.
sipri = sipri_raw.copy()
sipri = sipri.rename(columns={sipri.columns[0]: 'Country'})
sipri = sipri.loc[:, ~sipri.columns.astype(str).str.contains('^Unnamed')]

# Replace non-numeric placeholders used by SIPRI.
sipri = sipri.replace({'...': np.nan, 'xxx': np.nan, '. .': np.nan})

# Identify all year columns automatically.
sipri_year_cols = [col for col in sipri.columns if str(col).isdigit()]
sipri[sipri_year_cols] = sipri[sipri_year_cols].apply(pd.to_numeric, errors='coerce')

# Remove unnecessary rows such as region headings by keeping only rows that contain at least one numeric year value.
sipri = sipri[sipri[sipri_year_cols].notna().any(axis=1)].copy()
sipri = sipri[['Country'] + sipri_year_cols].copy()
sipri['Country'] = sipri['Country'].astype(str).str.strip()

# Convert from wide format to long format using melt.
sipri_long = sipri.melt(
    id_vars='Country',
    value_vars=sipri_year_cols,
    var_name='Year',
    value_name='Military_Spending'
)

sipri_long['Year'] = sipri_long['Year'].astype(int)
sipri_long = sipri_long.dropna(subset=['Military_Spending']).reset_index(drop=True)

display(sipri_long.head())
print('SIPRI long shape:', sipri_long.shape)

,Country,Year,Military_Spending
0,Costa Rica,1949,0.00
1,Mexico,1949,34.80
2,United States of America,1949,"14,088.16"
3,Argentina,1949,0.00
4,Peru,1949,23.90


SIPRI long shape: (8846, 3)


In [4]:
# -----------------------------
# 2. Clean GDP data
# -----------------------------

# The GDP file contains many metadata columns before the yearly columns.
# Keep only the country label plus the year columns and then melt it to long format.
gdp = gdp_raw.copy()
gdp_year_cols = [col for col in gdp.columns if str(col).isdigit()]
gdp = gdp[['REF_AREA_LABEL'] + gdp_year_cols].rename(columns={'REF_AREA_LABEL': 'Country'})

gdp_long = gdp.melt(
    id_vars='Country',
    value_vars=gdp_year_cols,
    var_name='Year',
    value_name='GDP'
)

gdp_long['Country'] = gdp_long['Country'].astype(str).str.strip()
gdp_long['Year'] = gdp_long['Year'].astype(int)
gdp_long['GDP'] = pd.to_numeric(gdp_long['GDP'], errors='coerce')
gdp_long = gdp_long.dropna(subset=['GDP']).reset_index(drop=True)

display(gdp_long.head())
print('GDP long shape:', gdp_long.shape)

,Country,Year,GDP
0,Syrian Arab Republic,1960,"857,704,413.41"
1,Low & middle income,1960,"266,021,244,949.63"
2,Morocco,1960,"2,037,154,741.93"
3,Botswana,1960,"30,411,413.66"
4,Middle income,1960,"254,380,808,746.11"


GDP long shape: (14561, 3)
